<a href="https://colab.research.google.com/github/uzairmustafa291/Flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uzairmustafa291/Flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

#what does one row represents:
One row represents the daily performance metrics of one content page (content_hash_id) for one client (client_hash_id) on one reporting date (report_date).

In [3]:
import pandas as pd
import numpy as np
from huggingface_hub import login


login()

In [21]:
from itertools import islice


In [10]:
from huggingface_hub import HfApi

api = HfApi()

repo_info = api.dataset_info("FlyRank/internship-warehouse")

print(repo_info)

DatasetInfo(id='FlyRank/internship-warehouse', author='FlyRank', card_data={'annotations_creators': None, 'language_creators': None, 'language': ['en'], 'license': 'other', 'multilinguality': None, 'size_categories': ['10M<n<100M'], 'source_datasets': None, 'task_categories': None, 'task_ids': None, 'paperswithcode_id': None, 'pretty_name': 'FlyRank Internship — Warehouse Star Schema (Pseudonymized, Gated)', 'config_names': None, 'train_eval_index': None, 'tags': ['seo', 'content-performance', 'data-warehouse', 'tabular', 'education', 'flyrank-internship'], 'extra_gated_prompt': 'By requesting access you agree to the FlyRank Internship Data Use Terms: anonymized research and education use only; no attempt to re-identify clients, domains, queries, keywords, or content; no redistribution of the raw data; and no client-identifying data in any public output (case study, repo, chart, or demo).', 'extra_gated_fields': {'Name': 'text', 'Email': 'text', 'Affiliation or cohort': 'text', 'I agre

In [18]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [23]:
sample = list(islice(ds, 5))
df=pd.DataFrame(sample)
df.head(2)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

#Features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'sessions_organic']
#Label:
sessions_ai
#Context:
['report_date', 'client_hash_id', 'content_hash_id']
#Excluded:
['ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other']

In [26]:
# Features
X = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "sessions_organic"
]

# Target
y = "sessions_ai"

# Context
context = [
    "report_date",
    "client_hash_id",
    "content_hash_id"
]

# Excluded
excluded = [
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other"
]

print("Features:", X)
print("Label:", y)
print("Context:", context)
print("Excluded:", excluded)

Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'sessions_organic']
Label: sessions_ai
Context: ['report_date', 'client_hash_id', 'content_hash_id']
Excluded: ['ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [27]:
# Verify that each (report_date, client_hash_id, content_hash_id) combination is unique

grain = (
    df.groupby(
        ["report_date", "client_hash_id", "content_hash_id"]
    )
    .size()
)

print(grain.value_counts())

1    5
Name: count, dtype: int64


In [28]:
print("Total Rows:", len(df))
print("Start Date:", df["report_date"].min())
print("End Date:", df["report_date"].max())

Total Rows: 5
Start Date: 2025-01-27
End Date: 2025-01-27


In [29]:
cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "sessions_organic",
    "sessions_ai"
]

df[cols].isnull().sum()

,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_pageviews,0
sessions_organic,0
sessions_ai,0


In [30]:
df["gsc_data_available"].value_counts()

,count
gsc_data_available,
True,5


In [32]:
X = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "sessions_organic"
]

feature_df = df[X]

feature_df.head(2)

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,sessions_organic
0,30,0,3.833333,0,0
1,5,0,71.600000,0,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

#Data limitations

The dataset uses hashed client and content IDs, so it does not provide information about the actual website, page topic, or industry.
Some rows have missing or unavailable GSC/GA4 data (gsc_data_available or ga4_data_available is False), reducing the amount of usable information.
The analysis is limited to a selected time window (a mid-panel month), so it may not capture seasonal or long-term trends.
External factors such as Google algorithm updates, marketing campaigns, or news events are not included in the dataset and therefore cannot be modeled.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.